In [28]:

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import re



In [19]:
driver = webdriver.Chrome()
driver.get("https://www.amazon.com/s?k=gaming+laptops&language=ar&_encoding=UTF8&pf_rd_p=3b3fe632-032a-4156-ba0f-c16b54ab9f0a&pf_rd_r=XS0GH0SD7AFSN07K9SAV&ref=cct_cg_purimil_2d1")
time.sleep(3)

In [29]:
data =[]


In [30]:
for page in range(1,20):
    url = f"https://www.amazon.eg/s?k=gaming+laptops&page={page}"
    driver.get(url)
    time.sleep(3)
    
    products = driver.find_elements(By.XPATH,"//div[@data-component-type='s-search-result' and @data-asin]")
    for product in products:
        try:
            title =product.find_element(By.XPATH, ".//h2").text
        except:
            title = None    
        try:
            price =product.find_element(By.XPATH, ".//span[@class='a-price-whole']").text
        except:
            price = None
        try:
            link =product.find_element(By.XPATH, ".//a[contains(@class,'a-link-normal')]").get_attribute("href")
        except:
            link = None 
        try:
            rating_el = product.find_element(By.XPATH,".//i[contains(@class,'a-icon-star-mini')]//span[contains(@class,'a-icon-alt')]" )
            rating_text = rating_el.text.strip()
            rating = re.search(r'[\d.]+', rating_text).group()
    
        except Exception as e:
            print("Rating error:", e)
            rating = None
           
            
        data.append({
            "title" :title,
            "price":price,
            "rating":rating,
            "link":link
        })                    
    


Rating error: 'NoneType' object has no attribute 'group'
Rating error: Message: no such element: Unable to locate element: {"method":"xpath","selector":".//i[contains(@class,'a-icon-star-mini')]//span[contains(@class,'a-icon-alt')]"}
  (Session info: chrome=131.0.6778.265); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	GetHandleVerifier [0x00007FF60F2580D5+2992373]
	(No symbol) [0x00007FF60EEEBFD0]
	(No symbol) [0x00007FF60ED8590A]
	(No symbol) [0x00007FF60EDD926E]
	(No symbol) [0x00007FF60EDD955C]
	(No symbol) [0x00007FF60EDCC6CC]
	(No symbol) [0x00007FF60EDFF3AF]
	(No symbol) [0x00007FF60EDCC596]
	(No symbol) [0x00007FF60EDFF580]
	(No symbol) [0x00007FF60EE1F584]
	(No symbol) [0x00007FF60EDFF113]
	(No symbol) [0x00007FF60EDCA918]
	(No symbol) [0x00007FF60EDCBA81]
	GetHandleVerifier [0x00007FF60F2B6A2D+3379789]
	GetHandleVerifier [0x00007FF60F2CC32D+3468109]
	GetHandleVerifier 

In [46]:
df =pd.DataFrame(data)
print(df.head())

                                               title   price rating  \
0  Lenovo LOQ 15IAX9 Gaming Laptop, Intel Core i5...  33,499   None   
1  Lenovo Legion 5 15IRX10 Gaming Laptop Intel Co...  80,399   None   
2  Lenovo LOQ 15IAX9 Gaming Laptop, Intel Core i5...  40,399   None   
3  Lenovo LOQ 15AHP10, AMD Ryzen™ 7 250, 16GB RAM...  59,999   None   
4  ASUS TUF F16 [FX607VJB-RL165W] Intel Core 5-21...  46,051   None   

                                                link  
0  https://www.amazon.eg/-/en/Lenovo-15IAX9-Gamin...  
1  https://www.amazon.eg/-/en/Lenovo-15IRX10-i7-1...  
2  https://www.amazon.eg/-/en/Lenovo-15IAX9-Gamin...  
3  https://www.amazon.eg/-/en/Lenovo-15AHP10-Ryze...  
4  https://www.amazon.eg/-/en/ASUS-FX607VJB-RL165...  


In [47]:
df.head

<bound method NDFrame.head of                                                  title   price rating  \
0    Lenovo LOQ 15IAX9 Gaming Laptop, Intel Core i5...  33,499   None   
1    Lenovo Legion 5 15IRX10 Gaming Laptop Intel Co...  80,399   None   
2    Lenovo LOQ 15IAX9 Gaming Laptop, Intel Core i5...  40,399   None   
3    Lenovo LOQ 15AHP10, AMD Ryzen™ 7 250, 16GB RAM...  59,999   None   
4    ASUS TUF F16 [FX607VJB-RL165W] Intel Core 5-21...  46,051   None   
..                                                 ...     ...    ...   
289             Fantech P51 5-in-1 PC Gaming Combo Set   1,899   None   
290  Redragon GM300 USB Gaming Microphone, Computer...   2,888   None   
291  MSI CLUTCHGM50 Gaming USB RGB Adjustable up to...   2,499   None   
292  EKSA E1000 USB Gaming Headset for PC - Compute...   3,999   None   
293  Onkuma X10 Oxhorn RGB Wired Gaming Headset, No...     975   None   

                                                  link  
0    https://www.amazon.eg/-/en/Leno

In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   294 non-null    str   
 1   price   277 non-null    str   
 2   rating  0 non-null      object
 3   link    294 non-null    str   
dtypes: object(1), str(3)
memory usage: 9.3+ KB


In [48]:
import re

def parse_title(title):
    result = {
        "brand": None,
        "series": None,
        "cpu": None,
        "ram": None,
        "storage": None,
        "gpu": None,
        "screen": None,
        "refresh_rate": None,
        "os": None,
        "color_from_title": None,
        "product_name": None
    }

    if not isinstance(title, str):
        return result

    t = title.strip()

    # product name
    result["product_name"] = t.split(",")[0].strip()

    # brand
    brands = ["Lenovo", "ASUS", "HP", "Dell", "GIGABYTE", "Acer", "MSI"]
    for b in brands:
        if re.search(rf"\b{re.escape(b)}\b", t, re.IGNORECASE):
            result["brand"] = b
            break

    # series
    series_patterns = ["Legion", "LOQ", "TUF", "ROG", "Victus", "Omen", "Alienware", "Strix"]
    for s in series_patterns:
        if re.search(rf"\b{re.escape(s)}\b", t, re.IGNORECASE):
            result["series"] = s
            break

    # cpu
    cpu_patterns = [
        r'Intel\s+Core\s+i[3579][-\s]?\d+[A-Z]*',
        r'Intel\s+i[3579][-\s]?\d+[A-Z]*',
        r'Core\s+Ultra\s*\d[-\s]?\d+[A-Z]*',
        r'Core\s+\d[-\s]?\d+[A-Z]*',
        r'Ryzen[™\s]*\d[\s-]*\d+[A-Z]*',
        r'Ryzen[™\s]*\d'
    ]
    for p in cpu_patterns:
        m = re.search(p, t, re.IGNORECASE)
        if m:
            result["cpu"] = m.group(0)
            break

    # ram
    ram_patterns = [
        r'\d+\s*GB\s*RAM',
        r'\d+\s*GB\s*DDR[45X]*',
        r'\d+\s*GB\s*LPDDR[45X]*',
        r'\d+\s*GB'
    ]
    for p in ram_patterns:
        m = re.search(p, t, re.IGNORECASE)
        if m:
            result["ram"] = m.group(0)
            break

    # storage
    storage_patterns = [
        r'\d+\s*TB\s*SSD',
        r'\d+\s*GB\s*SSD',
        r'\d+\s*TB',
        r'\d+\s*GB'
    ]
    for p in storage_patterns:
        m = re.search(p, t, re.IGNORECASE)
        if m and ("SSD" in m.group(0).upper() or "TB" in m.group(0).upper()):
            result["storage"] = m.group(0)
            break

    # gpu
    gpu_patterns = [
        r'RTX[™\s]*\d{4}\s*\d*\s*GB',
        r'GeForce\s+RTX\s*\d{4}',
        r'RTX\s*\d{4}',
        r'RTX™\s*\d{4}'
    ]
    for p in gpu_patterns:
        m = re.search(p, t, re.IGNORECASE)
        if m:
            result["gpu"] = m.group(0)
            break

    # screen
    screen_patterns = [
        r'\d{1,2}\.?\d?\s*["”]',
        r'\d{1,2}\.?\d?\s*inch',
        r'\d{1,2}\.?\d?["”]'
    ]
    for p in screen_patterns:
        m = re.search(p, t, re.IGNORECASE)
        if m:
            result["screen"] = m.group(0)
            break

    # refresh rate
    m = re.search(r'\d+\s*Hz', t, re.IGNORECASE)
    if m:
        result["refresh_rate"] = m.group(0)

    # os
    os_patterns = [
        r'Windows\s*11\s*Home',
        r'Windows\s*11',
        r'Win\s*11',
        r'Win11',
        r'Free\s*DOS',
        r'FreeDOS',
        r'\bDOS\b'
    ]
    for p in os_patterns:
        m = re.search(p, t, re.IGNORECASE)
        if m:
            result["os"] = m.group(0)
            break

    # color
    color_patterns = [
        "Luna Grey", "Eclipse Black", "Indigo",
        "Black", "Grey", "Gray", "White", "Blue"
    ]
    for c in color_patterns:
        if c.lower() in t.lower():
            result["color_from_title"] = c
            break

    return result

In [49]:
parsed = df["title"].apply(parse_title)

In [50]:
parsed_df = pd.DataFrame(parsed.tolist())

In [51]:
df = pd.concat([df, parsed_df], axis=1)

In [52]:
df["product_name"] = df["title"].apply(lambda x: x.split(",")[0] if isinstance(x, str) else None)

In [53]:
df.head()

,title,price,rating,link,brand,series,cpu,ram,storage,gpu,screen,refresh_rate,os,color_from_title,product_name
0,"Lenovo LOQ 15IAX9 Gaming Laptop, Intel Core i5...","33,499",None,https://www.amazon.eg/-/en/Lenovo-15IAX9-Gamin...,Lenovo,LOQ,Intel Core i5 12450HX,12GB RAM,512GB SSD,RTX 2050 4GB,"15.6""",144Hz,Win11,Luna Grey,Lenovo LOQ 15IAX9 Gaming Laptop
1,Lenovo Legion 5 15IRX10 Gaming Laptop Intel Co...,"80,399",None,https://www.amazon.eg/-/en/Lenovo-15IRX10-i7-1...,Lenovo,Legion,Intel Core i7-13650HX,16GB RAM,1TB SSD,RTX 5060 8GB,NaN,165Hz,DOS,NaN,Lenovo Legion 5 15IRX10 Gaming Laptop Intel Co...
2,"Lenovo LOQ 15IAX9 Gaming Laptop, Intel Core i5...","40,399",None,https://www.amazon.eg/-/en/Lenovo-15IAX9-Gamin...,Lenovo,LOQ,Intel Core i5 12600HX,16GB RAM,512GB SSD,RTX 3050 6GB,"15.6""",144Hz,Win11,Grey,Lenovo LOQ 15IAX9 Gaming Laptop
3,"Lenovo LOQ 15AHP10, AMD Ryzen™ 7 250, 16GB RAM...","59,999",None,https://www.amazon.eg/-/en/Lenovo-15AHP10-Ryze...,Lenovo,LOQ,Ryzen™ 7 250,16GB RAM,512GB SSD,RTX 5060 8GB,"15.6""",144Hz,DOS,Luna Grey,Lenovo LOQ 15AHP10
4,ASUS TUF F16 [FX607VJB-RL165W] Intel Core 5-21...,"46,051",None,https://www.amazon.eg/-/en/ASUS-FX607VJB-RL165...,ASUS,TUF,Core 5-210H,16GB DDR5,512GB SSD,RTX 3050 6GB,NaN,144Hz,Win 11,NaN,ASUS TUF F16 [FX607VJB-RL165W] Intel Core 5-21...


In [ ]:

df = df[[
    "product_name", "brand", "series", "cpu", "ram",
    "storage", "gpu", "screen", "refresh_rate", "os",
    "color_from_title", "price", "link"
]]

In [57]:
df.to_csv('amazon.csv', index=False)